# Dataset Generation for N-Body System Prediction

In [16]:
import numpy as np
import random as rd
import pandas as pd
import os
import sys
import time
import datetime
from math import *
from google.colab import drive, files

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
G_REAL = 6.6743e-11
G = 1

MU = 1e30
AU = 149597870700 # ​≈ 1.5e11

TU = sqrt(AU**3 / (G_REAL * MU)) # ​≈ 7 082 478 seconds ≈ 82 days
TU_AU = TU / AU

epsilon = 0.05
dt = 0.001

METHODS = {
    0: "Kepler 2-Body",
    1: "Semi-Implicit Euler",
    2: "Velocity Verlet",
    3: "Runge-Kutta 4"
}

METHODS_SHORT = {
    "Kepler 2-Body" : "kepler",
    "Semi-Implicit Euler" : "euler",
    "Velocity Verlet" : "verlet",
    "Runge-Kutta 4" : "rk4"
}

## Presets

In [ ]:
DEFAULT_N = 3
DEFAULT_STEPS = 2000
DEFAULT_N_SIM = 500
DEFAULT_OPTION = 3

user_input = input("Number of bodies: ").strip()
N = int(user_input) if user_input else DEFAULT_N
print("N =", N)

if (N > 10) or (N < 2):
    raise ValueError("Number of bodies must be between 2 and 10.")

user_input = input("Number of steps per simulation: ").strip()
steps = int(user_input) if user_input else DEFAULT_STEPS
print("steps =", steps)

if (steps < 0) or (steps > 10000):
    raise ValueError("Number of steps is a positive integer lower than 10 000.")

user_input = input("Number of independent simulations: ").strip()
n_sim = int(user_input) if user_input else DEFAULT_N_SIM
print("n_sim =", n_sim)

if (n_sim < 0) or (n_sim > 10000):
    raise ValueError("Number of simulations is a positive integer lower than 10 000.")

if N == 2:
    prompt = "Method: 0 = Kepler, 1 = Euler, 2 = Verlet, 3 = RK4: "
else:
    current_default = DEFAULT_OPTION if DEFAULT_OPTION != 0 else 3
    prompt = "Method: 1 = Euler, 2 = Verlet, 3 = RK4: "

user_input = input(prompt).strip()
option = int(user_input) if user_input else (DEFAULT_OPTION if (N == 2 or DEFAULT_OPTION != 0) else 3)

if option not in METHODS or (N != 2 and option == 0):
    raise ValueError("Invalid method for this number of bodies!")

print("Selected method:", METHODS[option])

## Classes

In [ ]:
class Body:
  def __init__(self, mass, position, velocity):
    self.mass = float(mass)
    self.position = np.array(position, dtype=float)
    self.velocity = np.array(velocity, dtype=float)
    self.acceleration = np.zeros(2, dtype=float)

  def __repr__(self):
    pos_str = np.array2string(self.position, formatter={'float_kind':lambda x: f"{x:.6f}"})
    vel_str = np.array2string(self.velocity, formatter={'float_kind':lambda x: f"{x:.6f}"})
    acc_str = np.array2string(self.acceleration, formatter={'float_kind':lambda x: f"{x:.6f}"})

    return (
        f"mass: {self.mass:.6f}, "
        f"pos: {pos_str}, "
        f"vel: {vel_str}, "
        f"acc: {acc_str}"
    )

In [ ]:
class KeplerBody(Body):
  def __init__(self, mass, position, velocity):
    super().__init__(mass, position, velocity)
    self.mu = 0
    self.a = 0
    self.e = 0
    self.n = 0
    self.M0 = 0
    self.arg_perigee = 0

  def init_orbit(self, central_body):
    r_vec = self.position - central_body.position
    v_vec = self.velocity - central_body.velocity

    r_mag = np.linalg.norm(r_vec)
    v_mag_sq = np.dot(v_vec, v_vec)

    self.mu = central_body.mass + self.mass
    self.a = 1.0 / ((2.0 / r_mag) - (v_mag_sq / self.mu))

    rdotv = np.dot(r_vec, v_vec)

    v2_mur = v_mag_sq - self.mu / r_mag

    e_vec = (v2_mur * r_vec - rdotv * v_vec) / self.mu
    self.e = np.linalg.norm(e_vec)

    self.arg_perigee = atan2(e_vec[1], e_vec[0])
    self.n = sqrt(self.mu / (self.a ** 3))

    cos_nu = np.clip(np.dot(e_vec, r_vec) / (self.e * r_mag), -1.0, 1.0)

    nu0 = np.arccos(cos_nu)
    if rdotv < 0:
        nu0 = 2 * pi - nu0

    E0 = 2 * atan2(
        sqrt(1 - self.e**2) * sin(nu0),
        1 - self.e * cos(nu0)
    )

    self.M0 = E0 - self.e * sin(E0)

  def step_kepler(self, t, central_body):
    M = (self.M0 + self.n * t) % (2 * pi)

    E_val = M
    delta = 1

    for _ in range(50):
      f = E_val - self.e * sin(E_val) - M
      f_prime = 1 - self.e * cos(E_val)

      delta = f / f_prime
      E_val -= delta

      if abs(delta) < 1e-12: break

    x_p = self.a * (cos(E_val) - self.e)
    y_p = self.a * sqrt(1 - self.e**2) * sin(E_val)

    r_inst = self.a * (1 - self.e * cos(E_val))
    v_factor = sqrt(self.mu * self.a) / r_inst

    vx_p = -v_factor * sin(E_val)
    vy_p = v_factor * sqrt(1 - self.e**2) * cos(E_val)

    cos_ap = cos(self.arg_perigee)
    sin_ap = sin(self.arg_perigee)

    x_rel = x_p * cos_ap - y_p * sin_ap
    y_rel = x_p * sin_ap + y_p * cos_ap

    vx_rel = vx_p * cos_ap - vy_p * sin_ap
    vy_rel = vx_p * sin_ap + vy_p * cos_ap

    Mtot = self.mass + central_body.mass

    self.position = np.array([
        -(central_body.mass / Mtot) * x_rel,
        -(central_body.mass / Mtot) * y_rel
    ])

    central_body.position = np.array([
        (self.mass / Mtot) * x_rel,
        (self.mass / Mtot) * y_rel
    ])

    self.velocity = np.array([
        -(central_body.mass / Mtot) * vx_rel,
        -(central_body.mass / Mtot) * vy_rel
    ])

    central_body.velocity = np.array([
        (self.mass / Mtot) * vx_rel,
        (self.mass / Mtot) * vy_rel
    ])

In [ ]:
class EulerBody(Body):
  def update_accelerations(self, others):
    accelerations = np.zeros(2, dtype=float)

    for other in others:
      if other is self: continue

      r_vec = other.position - self.position
      r_mag = np.linalg.norm(r_vec)

      accelerations += other.mass * r_vec / (r_mag**2 + epsilon**2)**1.5

    self.acceleration[:] = accelerations

  def update_velocity_positions(self, dt):
    # Semi-Implicit Euler (Euler-Cromer) Method
    self.velocity += self.acceleration * dt
    self.position += self.velocity * dt

In [ ]:
class VerletBody(Body):
  def __init__(self, mass, position, velocity):
    super().__init__(mass, position, velocity)
    self.old_acceleration = np.zeros(2, dtype=float)

  def update_acceleration(self, others):
    # store old acceleration
    self.old_acceleration = self.acceleration.copy()
    acc = np.zeros(2, dtype=float)
    for other in others:
      if other is self: continue
      r_vec = other.position - self.position
      r_sq_eps = np.dot(r_vec, r_vec) + epsilon**2
      acc += other.mass * r_vec / (r_sq_eps ** 1.5)
    self.acceleration = acc

  def update_velocity(self, dt):
    self.velocity += 0.5 * (self.old_acceleration + self.acceleration) * dt

  def update_position(self, dt):
    self.position += (self.velocity * dt + 0.5 * self.acceleration * dt**2)

In [ ]:
class RK4Body(Body):
  def __init__(self, mass, position, velocity):
    super().__init__(mass, position, velocity)
    # k vectors (velocity + acceleration samples)
    self.kv = [np.zeros(2) for _ in range(4)]
    self.ka = [np.zeros(2) for _ in range(4)]
    # temporary states
    self.temp_pos = np.zeros(2)
    self.temp_vel = np.zeros(2)

  def get_acceleration(self, others):
    acc = np.zeros(2)
    for other in others:
      if other is self: continue
      r_vec = other.temp_pos - self.temp_pos
      r2_epsilon = np.dot(r_vec, r_vec) + epsilon**2
      acc += other.mass * r_vec / (r2_epsilon ** 1.5)
    return acc

## Functions

### Dataset Functions

In [ ]:
header = ['id_sim', 'time']
for i in range(N):
  header.extend([f'm{i+1}', f'x{i+1}', f'y{i+1}', f'vx{i+1}', f'vy{i+1}'])
buffer = []

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
name = METHODS_SHORT.get(METHODS.get(option), "unknown")
FILENAME = f"{name}_{N}body_{timestamp}.csv"

buffer_count = 1
sims_in_current_buffer = 0
buffer_start_time = None

def format_time(seconds):
  if seconds < 60:
    return f"{int(seconds)}s"
  minutes = int(seconds // 60)
  seconds = int(seconds % 60)
  return f"{minutes}m {seconds}s"

def draw_bar(current, total, start_time):
  bar_width = 30
  progress = current / total
  arrow_pos = int(progress * bar_width)

  if arrow_pos > 0:
    bar = "-" * (arrow_pos - 1) + ">" + "." * (bar_width - arrow_pos)
  else:
    bar = "." * bar_width

  elapsed_time = time.time() - start_time
  sims_per_sec = current / elapsed_time if elapsed_time > 0 else 0
  ms_per_sim = (elapsed_time / current) * 1000 if current > 0 else 0

  if sims_per_sec > 0:
    eta_seconds = (total - current) / sims_per_sec
    eta_str = format_time(eta_seconds)
  else:
    eta_str = "N/A"

  sys.stdout.write(
    f"\r{current}/{total} [{bar}] - "
    f"{int(ms_per_sim)}ms/sim - "
    f"ETA: {eta_str} "
  )
  sys.stdout.flush()

def reset_buffer():
  global buffer, buffer_count, sims_in_current_buffer, buffer_start_time
  buffer = []
  buffer_count += 1
  sims_in_current_buffer = 0
  buffer_start_time = time.time()

def save_to_buffer(id_sim, current_time):
  row = [id_sim, current_time]
  for body in bodies:
    row.extend([body.mass, body.position[0], body.position[1],
                body.velocity[0], body.velocity[1]])
  buffer.append(row)

def save_to_csv():
  df = pd.DataFrame(buffer, columns=header)
  file_exists = os.path.isfile(FILENAME)

  df.to_csv(FILENAME, mode='a', index=False, header=not file_exists)
  print(f"\n---> Data successfully flushed and appended to {FILENAME}")

### Simulation Functions

In [ ]:
def generate_bodies(option):
  bodies = []

  sum_vx = 0.0
  sum_vy = 0.0

  for i in range(N):
    mass = rd.uniform(0.5, 2)

    x_pos = rd.uniform(-1, 1)
    y_pos = rd.uniform(-1, 1)

    if i < N - 1:
      vx = rd.uniform(-2e4, 2e4) * TU_AU
      vy = rd.uniform(-2e4, 2e4) * TU_AU

      sum_vx += mass * vx
      sum_vy += mass * vy

    else:
      vx = -sum_vx / mass
      vy = -sum_vy / mass

    pos = np.array([x_pos, y_pos], dtype=float)
    vel = np.array([vx, vy], dtype=float)

    if option == 0: body = KeplerBody(mass, pos, vel)
    elif option == 1: body = EulerBody(mass, pos, vel)
    elif option == 2: body = VerletBody(mass, pos, vel)
    elif option == 3: body = RK4Body(mass, pos, vel)

    bodies.append(body)

  return bodies

In [ ]:
def run_kepler(steps, dt, bodies, id_sim):
  body1 = bodies[0]
  body2 = bodies[1]

  body1.init_orbit(body2)
  current_time = 0

  for s in range(steps):
    current_time += dt
    body1.step_kepler(current_time, body2)

    current_time_dataset = s * dt
    save_to_buffer(id_sim, current_time_dataset)

In [ ]:
def run_euler(steps, dt, bodies, id_sim):
  for s in range(steps):
    for body in bodies:
      body.update_accelerations(bodies)

    for body in bodies:
      body.update_velocity_positions(dt)

    current_time = s * dt
    save_to_buffer(id_sim, current_time)

In [ ]:
def run_verlet(steps, dt, bodies, id_sim):
  for s in range(steps):

    for body in bodies:
      body.update_position(dt)

    for body in bodies:
      body.update_acceleration(bodies)

    for body in bodies:
      body.update_velocity(dt)

    current_time = s * dt
    save_to_buffer(id_sim, current_time)

In [ ]:
def run_rk4(steps, dt, bodies, id_sim):
  for s in range(steps):

    # Step k1
    for b in bodies:
        b.temp_pos = b.position.copy()
    for b in bodies:
        b.kv[0] = b.velocity.copy()
        b.ka[0] = b.get_acceleration(bodies)

    # Step k2
    for b in bodies:
        b.temp_pos = b.position + b.kv[0] * dt * 0.5
        b.temp_vel = b.velocity + b.ka[0] * dt * 0.5

    for b in bodies:
        b.kv[1] = b.temp_vel.copy()
        b.ka[1] = b.get_acceleration(bodies)

    # Step k3
    for b in bodies:
        b.temp_pos = b.position + b.kv[1] * dt * 0.5
        b.temp_vel = b.velocity + b.ka[1] * dt * 0.5

    for b in bodies:
        b.kv[2] = b.temp_vel.copy()
        b.ka[2] = b.get_acceleration(bodies)

    # Step k4
    for b in bodies:
        b.temp_pos = b.position + b.kv[2] * dt
        b.temp_vel = b.velocity + b.ka[2] * dt

    for b in bodies:
        b.kv[3] = b.temp_vel.copy()
        b.ka[3] = b.get_acceleration(bodies)

    # Final Integration
    for b in bodies:
        b.position += (dt / 6.0) * (b.kv[0] + 2*b.kv[1] + 2*b.kv[2] + b.kv[3])
        b.velocity += (dt / 6.0) * (b.ka[0] + 2*b.ka[1] + 2*b.ka[2] + b.ka[3])

    current_time = s * dt
    save_to_buffer(id_sim, current_time)

## Simulation Loop

In [ ]:
buffer_start_time = time.time()
print(f"Data Generation Target: {n_sim} Simulations ({N}-Body RK4)")
print(f"Buffer Size: 100 simulations per block\n")
print(f"Buffer {buffer_count}/{(n_sim-1)//100 + 1}")

for id_sim in range(n_sim):
  # GENERATING BODIES
  bodies = generate_bodies(option)

  # SIMULATION LOOP
  if option == 0: run_kepler(steps, dt, bodies, id_sim)
  elif option == 1: run_euler(steps, dt, bodies, id_sim)
  elif option == 2: run_verlet(steps, dt, bodies, id_sim)
  elif option == 3: run_rk4(steps, dt, bodies, id_sim)

  sims_in_current_buffer += 1
  current_buffer_max = min(100, n_sim - ((buffer_count - 1) * 100))
  draw_bar(sims_in_current_buffer, current_buffer_max, buffer_start_time)

  if (id_sim + 1) % 100 == 0:
    save_to_csv()
    total_buffer_duration = time.time() - buffer_start_time

    reset_buffer()
    if (id_sim + 1) < n_sim:
      print(f"\nBuffer {buffer_count}/{(n_sim-1)//100 + 1}")

save_to_csv()
reset_buffer()

In [ ]:
save_dataset = input("Save the dataset? [y/n]: ")

if save_dataset.strip().lower() in ['y', 'yes']:

    print("Connecting to Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    drive_path = os.path.join('/content/drive/MyDrive/Colab Notebooks', FILENAME)
    import shutil
    shutil.copy(f'/content/{FILENAME}', drive_path)
    print(f"Successfully saved to Google Drive at: {drive_path}")

    print("Triggering browser download...")
    files.download(f'/content/{FILENAME}')

elif save_dataset.strip().lower() in ['n', 'no']:
    print("Dataset saving skipped.")

else:
    print("Invalid input. Proceeding without saving.")